In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import pg8000


In [ ]:
def process_games(games):
    processed = []
    
    for g in games:
        row = {}
        
        row["name"] = g.get("name")
        
        row["rating"] = g.get("rating")
        row["ratings_count"] = g.get("ratings_count")
        row["reviews_text_count"] = g.get("reviews_text_count")
        row["metacritic"] = g.get("metacritic")
        row["added"] = g.get("added")
        row["suggestions_count"] = g.get("suggestions_count")
        
        released = g.get("released")
        if released:
            parts = released.split("-")
            row["release_year"] = int(parts[0])
            row["release_month"] = int(parts[1])
        else:
            row["release_year"] = None
            row["release_month"] = None
        

        processed.append(row)
    
    return pd.DataFrame(processed)

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

conn = pg8000.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME", "postgres"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)
query = "SELECT * FROM juego"

df = pd.read_sql_query(query, conn)

In [ ]:
df["released"] = pd.to_datetime(df["released"], errors="coerce")

df["release_year"] = df["released"].dt.year
df["release_month"] = df["released"].dt.month

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=["rating", "ratings_count", "release_year"])

df["metacritic_missing"] = df["metacritic"].isna().astype(int)

df["metacritic"] = df["metacritic"].fillna(df["metacritic"].mean())
df = df[df["ratings_count"] > 50]

df = df.reset_index(drop=True)

df.shape

In [ ]:
df.head()

In [ ]:
df["genres"] = df["genres"].fillna("").apply(
    lambda x: [g for g in x.split(", ") if g != ""]
)
df["platforms"] = df["platforms"].fillna("").apply(lambda x: x.split(", "))

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
mlb_platforms = MultiLabelBinarizer()

platforms_encoded = pd.DataFrame(
    mlb_platforms.fit_transform(df["platforms"]),
    columns=["platform_" + p.lower().replace(" ", "_") for p in mlb_platforms.classes_]
)

platforms_encoded.head()

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb_genres = MultiLabelBinarizer()

genres_encoded = pd.DataFrame(
    mlb_genres.fit_transform(df["genres"]),
    columns=["genre_" + g.lower().replace(" ", "_") for g in mlb_genres.classes_]
)

genres_encoded.head()

In [ ]:
df = pd.concat([df, genres_encoded, platforms_encoded], axis=1)

df.drop(columns=["genres", "platforms"], inplace=True)

df.head()

In [ ]:
df.columns

In [ ]:
df["platform_pc_main"] = df.filter(like="platform_pc").sum(axis=1)
df["platform_playstation_main"] = df.filter(like="platform_playstation").sum(axis=1)
df["platform_xbox_main"] = df.filter(like="platform_xbox").sum(axis=1)
df["platform_nintendo_main"] = df.filter(like="platform_nintendo").sum(axis=1)

In [ ]:
platform_cols = [col for col in df.columns if col.startswith("platform_")]
df = df.drop(columns=platform_cols)

In [ ]:
df.columns

In [ ]:
df["success"] = ((df["rating"] >= 4) & (df["ratings_count"] > 500)).astype(int)

In [ ]:
df["success"].value_counts()

In [ ]:
X = df.drop(columns=[
    "success",
    "name",
    "rating",
    "ratings_count"
])

X = X.select_dtypes(include=["number", "bool"]).copy()

X = X.fillna(0)

y = df["success"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    min_samples_leaf=5
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
import pandas as pd

feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance.head(10)

In [ ]:
import matplotlib.pyplot as plt

feature_importance.head(10).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.show()

In [ ]:
results = X_test.copy()
results["name"] = df.loc[X_test.index, "name"]
results["real"] = y_test
results["pred"] = y_pred

results.head(10)

In [ ]:
df["metacritic"].isna().sum()

In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

def predecir_exito(selected_genres):
    
    input_data = pd.DataFrame(np.zeros((1, len(X.columns))), columns=X.columns)
    
    for genre in selected_genres:
        col_name = f"genre_{genre.lower().replace(' ', '_')}"
        if col_name in input_data.columns:
            input_data[col_name] = 1
    
    for col in ["reviews_text_count", "metacritic", "added", "suggestions_count"]:
        if col in input_data.columns:
            input_data["reviews_text_count"] = df["reviews_text_count"].quantile(0.75)
            input_data["metacritic"] = 85
            input_data["added"] = df["added"].quantile(0.75)
            input_data["suggestions_count"] = df["suggestions_count"].quantile(0.75)
    
    if "release_year" in input_data.columns:
        input_data["release_year"] = df["release_year"].mean()
        
    if "release_month" in input_data.columns:
        input_data["release_month"] = 6
    
    if "metacritic_missing" in input_data.columns:
        input_data["metacritic_missing"] = 0
    
    prob = model.predict_proba(input_data)[0][1]
    pred = 1 if prob > 0.4 else 0
    prob = model.predict_proba(input_data)[0][1]
    
    return pred, prob


genres = [col.replace("genre_", "").replace("_", " ").title() 
          for col in X.columns if col.startswith("genre_")]

genres_widgets = {g: widgets.Checkbox(description=g) for g in genres}

genres_box = widgets.VBox(list(genres_widgets.values()))

button = widgets.Button(
    description="Predecir",
    button_style="success"
)

output = widgets.Output()


def on_button_click(b):
    with output:
        clear_output()
        
        selected_genres = [g for g, w in genres_widgets.items() if w.value]
        
        if len(selected_genres) == 0:
            print(" Selecciona al menos un género")
            return
        
        pred, prob = predecir_exito(selected_genres)
        
        if pred == 1:
            print(f" Será un ÉXITO")
            print(f"Probabilidad: {prob:.2f}")
        else:
            print(f" No será un éxito")
            print(f"Probabilidad: {prob:.2f}")

button.on_click(on_button_click)


display(
    widgets.Label("Selecciona géneros:"),
    genres_box,
    button,
    output
)

In [ ]:
import joblib
from pathlib import Path

artifacts_dir = (Path("..") / ".." / "artifacts").resolve()
artifacts_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifacts_dir / "success_model.joblib")
joblib.dump(list(X.columns), artifacts_dir / "success_features.joblib")

stats = {
    "reviews_text_count_q75": float(df["reviews_text_count"].quantile(0.75)) if "reviews_text_count" in df.columns else 0.0,
    "added_q75": float(df["added"].quantile(0.75)) if "added" in df.columns else 0.0,
    "suggestions_count_q75": float(df["suggestions_count"].quantile(0.75)) if "suggestions_count" in df.columns else 0.0,
    "metacritic_default": 85.0,
    "release_year_mean": float(df["release_year"].mean()) if "release_year" in df.columns else 0.0,
    "release_month_default": 6.0,
    "threshold": 0.4,
}
joblib.dump(stats, artifacts_dir / "success_stats.joblib")

print("Guardado en:", artifacts_dir)
